# SENSERO — GeoPackage sanity check

Quick inspection of the SENSERO patch-footprint GeoPackage (`sensero_patch_footprints.gpkg`):
lists the layers, prints each layer's schema / CRS / extent, previews a few rows, and runs
basic sanity checks (feature count, expected `caption*` columns, geometry health, split distribution).

**Requirements** (WSL / Jupyter Lab): `geopandas` and `pandas`.
Install, e.g. `pip install geopandas` &mdash; the conda-forge channel is recommended for the GDAL stack:
`conda install -c conda-forge geopandas`.

## 1. Configuration

In [1]:
from pathlib import Path

# --- Configure for your WSL setup -------------------------------------------
# Point this at the SENSERO GeoPackage. Examples:
#   Path.home() / "SENSERO" / "sensero_patch_footprints.gpkg"            # Linux/WSL home
#   Path("/mnt/c/Users/<you>/Downloads/sensero_patch_footprints.gpkg")   # Windows drive from WSL
GPKG_PATH = Path("/home/ubuntu/SENSERO/sensero_patch_footprints.gpkg")
# ---------------------------------------------------------------------------

EXPECTED_FEATURES   = 10_000
EXPECTED_CRS_EPSG   = 4326                       # WGS-84
EXPECTED_LABEL_COLS = [f"caption{i}" for i in range(1, 6)]

if not GPKG_PATH.exists():
    raise FileNotFoundError(f"GeoPackage not found: {GPKG_PATH.resolve()}")

print(f"Inspecting : {GPKG_PATH.resolve()}")
print(f"File size  : {GPKG_PATH.stat().st_size / 1e6:.1f} MB")

Inspecting : /home/ubuntu/SENSERO/sensero_patch_footprints.gpkg
File size  : 9.0 MB


## 2. Layers in the GeoPackage

In [2]:
import geopandas as gpd
import pandas as pd
from IPython.display import display

# List layer names without loading geometry (pyogrio backend if available, else fiona)
try:
    from pyogrio import list_layers
    layer_names = [row[0] for row in list_layers(str(GPKG_PATH))]
except Exception:
    import fiona
    layer_names = list(fiona.listlayers(str(GPKG_PATH)))

print(f"{len(layer_names)} layer(s):")
for name in layer_names:
    print(f"  - {name}")

1 layer(s):
  - patches


ERROR 1: libpoppler.so.140: cannot open shared object file: No such file or directory
ERROR 1: libpoppler.so.140: cannot open shared object file: No such file or directory


## 3. Per-layer schema, CRS and extent

In [3]:
def describe_layer(path, layer):
    gdf = gpd.read_file(path, layer=layer)
    print("=" * 72)
    print(f"Layer    : {layer}")
    print(f"Features : {len(gdf):,}")
    print(f"Geometry : {sorted(gdf.geom_type.dropna().unique())}")
    print(f"CRS      : {gdf.crs}")
    minx, miny, maxx, maxy = gdf.total_bounds
    print(f"Bounds   : x [{minx:.4f}, {maxx:.4f}]   y [{miny:.4f}, {maxy:.4f}]")
    print(f"Columns  ({len(gdf.columns)}):")
    for col, dtype in gdf.dtypes.items():
        print(f"    {col:<28} {dtype}")
    return gdf

layers = {name: describe_layer(GPKG_PATH, name) for name in layer_names}

Layer    : patches
Features : 10,000
Geometry : ['Polygon']
CRS      : EPSG:4326
Bounds   : x [20.3111, 29.6651]   y [43.6350, 48.2395]
Columns  (10):
    filename                     object
    rel_path                     object
    caption1                     object
    caption2                     object
    caption3                     object
    caption4                     object
    caption5                     object
    CLC_codes                    object
    split                        object
    geometry                     geometry


## 4. Row preview

In [4]:
# Detailed preview of the first (patch-footprint) layer
gdf = layers[layer_names[0]]
with pd.option_context("display.max_colwidth", 70, "display.width", 180):
    display(gdf.drop(columns=gdf.geometry.name).head(3))

,filename,rel_path,caption1,caption2,caption3,caption4,caption5,CLC_codes,split
0,S2A_MSIL2A_20181014T093031_N0500_R136_T34TFS_10056_3577,Multispectral/S2A_MSIL2A_20181014T093031_N0500_R136_T34TFS/S2A_MSI...,"artificial surfaces, agricultural areas, forest and semi-natural z...","industrial commercial and transport units, pastures, arable land, ...","road and rail networks and associated land, pastures, non-irrigate...","pastures, arable land, broad-leaved forest, complex cultivation pa...","pastures, agriculture, forests, mixed farmland, urban fabric","211, 231, 311, 122, 242, 112, 243",train
1,S2A_MSIL2A_20181014T093031_N0500_R136_T34TFS_10062_1155,Multispectral/S2A_MSIL2A_20181014T093031_N0500_R136_T34TFS/S2A_MSI...,"artificial surfaces, agricultural areas","urban fabric, arable land, pastures, permanent crops, heterogeneou...","discontinuous urban fabric, non-irrigated arable land, pastures, f...","urban fabric, arable land, pastures, permanent crops, land princip...","urban fabric, agriculture, pastures, permanent crops, mixed farmla...","211, 222, 231, 112, 133, 243, 242",val
2,S2A_MSIL2A_20181014T093031_N0500_R136_T34TFS_10093_8416,Multispectral/S2A_MSIL2A_20181014T093031_N0500_R136_T34TFS/S2A_MSI...,"agricultural areas, artificial surfaces","arable land, urban fabric","non-irrigated arable land, discontinuous urban fabric","arable land, urban fabric","agriculture, urban fabric","211, 112",train


## 5. Sanity checks

In [5]:
def check(label, ok, detail=""):
    tag = "PASS" if ok else "FAIL"
    print(f"[{tag}] {label}" + (f"  ->  {detail}" if detail else ""))
    return bool(ok)

gdf  = layers[layer_names[0]]
cols = set(gdf.columns)

# feature count
check(f"feature count == {EXPECTED_FEATURES:,}", len(gdf) == EXPECTED_FEATURES, f"got {len(gdf):,}")

# CRS
epsg = gdf.crs.to_epsg() if gdf.crs is not None else None
check(f"CRS is EPSG:{EXPECTED_CRS_EPSG} (WGS-84)", epsg == EXPECTED_CRS_EPSG, f"got EPSG:{epsg}")

# extent plausibly over Romania (WGS-84 lon/lat)
minx, miny, maxx, maxy = gdf.total_bounds
in_box = (19 <= minx <= 31) and (19 <= maxx <= 31) and (43 <= miny <= 49) and (43 <= maxy <= 49)
check("extent within Romania bounding box", in_box,
      f"bounds=[{minx:.2f}, {miny:.2f}, {maxx:.2f}, {maxy:.2f}]")

# expected label columns
missing = [c for c in EXPECTED_LABEL_COLS if c not in cols]
check("caption1-caption5 present", not missing,
      ("missing: " + ", ".join(missing)) if missing else "all present")

# geometry health
n_empty   = int(gdf.geometry.is_empty.sum())
n_invalid = int((~gdf.geometry.is_valid).sum())
check("no empty geometries", n_empty == 0, f"{n_empty} empty")
check("all geometries valid", n_invalid == 0, f"{n_invalid} invalid")

# duplicate footprints (patch centers are shared across scales, but within one layer they should be unique)
n_dup = int(gdf.geometry.duplicated().sum())
check("no duplicate footprints", n_dup == 0, f"{n_dup} duplicates")

# nulls in label columns
present = [c for c in EXPECTED_LABEL_COLS if c in cols]
nulls   = {c: int(gdf[c].isna().sum()) for c in present}
check("no nulls in caption columns", all(v == 0 for v in nulls.values()), str(nulls))

[PASS] feature count == 10,000  ->  got 10,000
[PASS] CRS is EPSG:4326 (WGS-84)  ->  got EPSG:4326
[PASS] extent within Romania bounding box  ->  bounds=[20.31, 43.64, 29.67, 48.24]
[PASS] caption1-caption5 present  ->  all present
[PASS] no empty geometries  ->  0 empty
[PASS] all geometries valid  ->  0 invalid
[PASS] no duplicate footprints  ->  0 duplicates
[PASS] no nulls in caption columns  ->  {'caption1': 0, 'caption2': 0, 'caption3': 0, 'caption4': 0, 'caption5': 0}


True

## 6. Split distribution

In [6]:
# Split distribution (column name may vary; the schema uses 'split')
split_col = next((c for c in ["split", "Split", "split1"] if c in gdf.columns), None)

if split_col is None:
    print("No split column found (looked for: split, Split, split1).")
else:
    total  = len(gdf)
    counts = gdf[split_col].value_counts(dropna=False)
    print(f"Split column: '{split_col}'  (expected ~70 / 15 / 15)")
    for value, n in counts.items():
        print(f"  {str(value):<12} {n:>6,}  ({n / total:6.1%})")

Split column: 'split'  (expected ~70 / 15 / 15)
  train         7,000  ( 70.0%)
  val           1,500  ( 15.0%)
  test          1,500  ( 15.0%)
